In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from mpetools import IslandTime, Segmentation, TimeSeriesPreProcessing
from coastsatmaster.coastsat import SDS_preprocess, SDS_tools
from celluloid import Camera
import geopandas as gpd
import shapely
import pyproj
import cv2

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
island = 'Vodamulaa'
country = 'Maldives'
island_info = IslandTime.retrieve_island_info(island, country, verbose=False)

lat = island_info['spatial_reference']['latitude']
lon = island_info['spatial_reference']['longitude']

metadata = pickle.load(open(os.path.join(os.getcwd(), 'data', 'coastsat_data', '{}_{}'.format(island, country), '{}_{}_metadata.pkl'.format(island, country)), 'rb'))

# Retrieve settings
settings = island_info['timeseries_coastsat']['settings']

idx_sat = 0

# Reference shoreline
reference_shoreline = island_info['spatial_reference']['reference_shoreline']

# Loop in every satellite
for sat in ['S2']:
    print(sat)
    buff = island_info['spatial_reference']['reference_shoreline_buffer_{}'.format(sat)]
    buff_nan = np.where(buff == 0., np.nan, buff)

    # Empty metadata for this satellite
    if metadata[sat]['filenames'] == []:
        continue

    # File path and names from CoastSat folder
    filepath = SDS_tools.get_filepath(settings['inputs'], sat)
    filenames = metadata[sat]['filenames']

    # Loop through images within the folder (taken from SDS_classify.py)
    for idx_img in range(len(filenames)):
        print('file # {}/{}'.format(idx_img+1, len(filenames)))



        str_filename = filenames[idx_img].split('-')
        year = str_filename[0]
        month = str_filename[1]
        day = str_filename[2]
        
        # Get file name
        fn = SDS_tools.get_filenames(filenames[idx_img], filepath, sat)

        # Retrieve information about image
        try:
            im_ms, georef, cloud_mask, _, _, im_nodata = SDS_preprocess.preprocess_single(fn, sat, settings['cloud_mask_issue'], settings['pan_off'], 'C02')
        except:
            continue
        # Compute cloud_cover percentage (with no data pixels)
        cloud_cover_combined = np.divide(sum(sum(cloud_mask.astype(int))), (cloud_mask.shape[0] * cloud_mask.shape[1]))
        
        # If 99% of cloudy pixels in image -> skip
        if cloud_cover_combined > 0.99: 
            continue
            
        image_epsg = metadata[sat]['epsg'][idx_img]
    
        # transform lat/lon
        src_crs = pyproj.CRS('EPSG:4326')
        tgt_crs = pyproj.CRS('EPSG:{}'.format(image_epsg))
        transformer = pyproj.Transformer.from_crs(src_crs, tgt_crs, always_xy=True)

        xx, yy = transformer.transform(lon, lat)

        idx_x = int((xx - georef[0]) / georef[1])
        idx_y = int((yy - georef[3]) / georef[5])

        center_of_image = shapely.geometry.Point(idx_x, idx_y)

        # Remove no data pixels from the cloud mask (for example L7 bands of no data should not be accounted for)
        cloud_mask_adv = np.logical_xor(cloud_mask, im_nodata)

        # Compute updated cloud cover percentage (without no data pixels)
        cloud_cover = np.divide(sum(sum(cloud_mask_adv.astype(int))), (sum(sum((~im_nodata).astype(int)))))

        # Skip image if cloud cover is above threshold
        if cloud_cover > 0.15 or cloud_cover == 1: #
            continue

        # Get images
        img_red = im_ms[:, :, 2]
        img_blue = im_ms[:, :, 0]
        img_green = im_ms[:, :, 1]
        img_nir = im_ms[:, :, 3]
        img_swir = im_ms[:, :, 4]

        # NDVI = (NIR - RED) / (NIR + RED)
        img_NDVI = SDS_tools.nd_index(im_ms[:, :, 3], im_ms[:, :, 2], cloud_mask)

        # MNWDI = (GREEN - SWIR) / (GREEN + SWIR)
        img_MNDWI = SDS_tools.nd_index(im_ms[:, :, 4], im_ms[:, :, 1], cloud_mask)

        # Stacked array
        img_stacked = np.dstack((img_red, 
                                 img_green,
                                 img_blue, 
                                 img_nir,
                                 img_swir, 
                                 img_NDVI,
                                 img_MNDWI))

        rgb = np.dstack((img_red * ~cloud_mask, img_green * ~cloud_mask, img_blue * ~cloud_mask))

        if np.char.startswith(filenames[idx_img], '2018-11-18-05-38-44_S2'):
            plt.imshow(rgb)
            plt.show()

            # Convert the image to grayscale
            gray = cv2.cvtColor(rgb.astype('float32'), cv2.COLOR_BGR2GRAY)

            # Apply edge detection using Canny
            edges = cv2.Canny(gray, 50, 150, apertureSize=3)

            # Use Hough Line Transform to detect lines
            lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=100)
            print(lines)
            break

S2
file # 1/408
file # 2/408
file # 3/408
file # 4/408
file # 5/408
file # 6/408
file # 7/408
file # 8/408
file # 9/408
file # 10/408
file # 11/408
file # 12/408
file # 13/408
file # 14/408
file # 15/408
file # 16/408
file # 17/408
file # 18/408
file # 19/408
file # 20/408
file # 21/408
file # 22/408
file # 23/408
file # 24/408
file # 25/408
file # 26/408
file # 27/408
file # 28/408
file # 29/408
file # 30/408
file # 31/408
file # 32/408
file # 33/408
file # 34/408
file # 35/408
file # 36/408
file # 37/408
file # 38/408
file # 39/408
file # 40/408
file # 41/408
file # 42/408
file # 43/408
file # 44/408
file # 45/408
file # 46/408
file # 47/408
file # 48/408
file # 49/408
file # 50/408
file # 51/408
file # 52/408
file # 53/408
file # 54/408
file # 55/408
file # 56/408
file # 57/408
file # 58/408
file # 59/408
file # 60/408
file # 61/408
file # 62/408
file # 63/408
file # 64/408
file # 65/408
file # 66/408
file # 67/408
file # 68/408
file # 69/408
file # 70/408
file # 71/408
file # 72/40

error: OpenCV(4.8.1) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\canny.cpp:829: error: (-215:Assertion failed) _src.depth() == CV_8U in function 'cv::Canny'


In [81]:
plt.hist(img_g.flatten(), bins=250)

(array([3.000e+00, 1.000e+02, 0.000e+00, 1.005e+03, 0.000e+00, 5.940e+03,
        0.000e+00, 1.229e+04, 5.651e+03, 0.000e+00, 1.758e+03, 0.000e+00,
        6.110e+02, 0.000e+00, 3.140e+02, 0.000e+00, 1.870e+02, 1.130e+02,
        0.000e+00, 6.400e+01, 0.000e+00, 6.500e+01, 0.000e+00, 2.000e+01,
        0.000e+00, 1.400e+01, 1.800e+01, 0.000e+00, 1.200e+01, 0.000e+00,
        1.400e+01, 0.000e+00, 1.100e+01, 1.900e+01, 0.000e+00, 1.400e+01,
        0.000e+00, 9.000e+00, 0.000e+00, 8.000e+00, 0.000e+00, 1.100e+01,
        8.000e+00, 0.000e+00, 9.000e+00, 0.000e+00, 2.000e+00, 0.000e+00,
        8.000e+00, 0.000e+00, 5.000e+00, 6.000e+00, 0.000e+00, 4.000e+00,
        0.000e+00, 7.000e+00, 0.000e+00, 8.000e+00, 9.000e+00, 0.000e+00,
        8.000e+00, 0.000e+00, 4.000e+00, 0.000e+00, 6.000e+00, 0.000e+00,
        7.000e+00, 9.000e+00, 0.000e+00, 6.000e+00, 0.000e+00, 1.200e+01,
        0.000e+00, 9.000e+00, 1.100e+01, 0.000e+00, 9.000e+00, 0.000e+00,
        1.600e+01, 0.000e+00, 1.400e+0

In [98]:
img_g = (img_nir*255).astype(np.uint8)
plt.imshow(img_g)

In [99]:
#gray = cv2.cvtColor(rgb.astype('float32'), cv2.COLOR_BGR2GRAY)

# Apply edge detection using Canny
edges = cv2.Canny(img_g, 0, 35, apertureSize=3)

# Use Hough Line Transform to detect lines
lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=20)
print(lines)

[[[  43.           1.8849555]]

 [[  23.           2.1816616]]

 [[  33.           1.9722221]]

 ...

 [[ -88.           2.8797932]]

 [[ -79.           2.9321532]]

 [[-119.           3.054326 ]]]


In [96]:
for line in lines:
    rho, theta = line[0]
    a = np.cos(theta)
    b = np.sin(theta)
    x0 = a*rho
    y0 = b*rho
    x1 = int(x0 + 1000*(-b))
    y1 = int(y0 + 1000*(a))
    x2 = int(x0 - 1000*(-b))
    y2 = int(y0 - 1000*(a))
    #cv2.line(img, (x1, y1), (x2, y2), (0, 0, 255), 2)



In [97]:
# Display the result
plt.subplot(121), plt.imshow(edges)
plt.title('Edge Image'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(img_g)
plt.title('Image with Lines'), plt.xticks([]), plt.yticks([])
plt.show()